# 🏎️ Autonomous Car Racing — Deep Q-Network (DQN)

**Author:** Herin Bhatt · MSc AI/ML Independent Project  
**Algorithm:** Deep Q-Network (Mnih et al., 2015)  
**Environment:** `CarRacing-v2` · OpenAI Gymnasium  
**Framework:** PyTorch · GPU (T4)

---

### Pipeline overview
| Step | Component | What it does |
|------|-----------|-------------|
| 1 | Dependencies | Install packages, start virtual display |
| 2 | Config & imports | All hyperparameters in one `CONFIG` dict |
| 3 | Frame preprocessing | Grayscale → resize → stack 4 frames |
| 4 | `DQNNetwork` | CNN: 4×84×84 frames → Q-values per action |
| 5 | `ReplayBuffer` | 50k-transition circular experience buffer |
| 6 | `DQNAgent` | ε-greedy selection + Bellman update + target net |
| 7 | `train()` | 1 000-episode loop with checkpointing & plots |
| 8 | `evaluate()` | Greedy policy benchmark (no exploration) |
| 9 | `record_video()` | MP4 of the trained agent driving |
| 10 | Download | Zip checkpoints + plots + video |

> **Runtime:** Enable GPU → Runtime ▸ Change runtime type ▸ **T4 GPU**  
> **Expected training time:** 3–5 hours · improvement visible from episode ~150

## Step 1 — Install dependencies

In [ ]:
# ── Packages ──────────────────────────────────────────────────────────────────
!pip install gymnasium[box2d] opencv-python-headless matplotlib torch torchvision -q
!apt-get install -y xvfb python3-opengl ffmpeg > /dev/null 2>&1
!pip install pyvirtualdisplay -q

# ── Virtual display (required for Colab rendering) ────────────────────────────
from pyvirtualdisplay import Display
Display(visible=0, size=(400, 300)).start()

print('✓ All dependencies installed.')

## Step 2 — Imports and configuration

In [ ]:
import os, random, math, time, collections
import numpy as np
import cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

import gymnasium as gym
from gymnasium.wrappers import RecordVideo

# ── Device ────────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}' + (f' — {torch.cuda.get_device_name(0)}' if device.type == 'cuda' else ''))

# ── Output directories ────────────────────────────────────────────────────────
for d in ('checkpoints', 'plots', 'videos'):
    os.makedirs(d, exist_ok=True)

# ── Hyperparameters (single source of truth) ──────────────────────────────────
CONFIG = {
    # Environment
    'env_name'           : 'CarRacing-v2',
    'frame_stack'        : 4,        # Stacked grayscale frames fed to the CNN
    'frame_size'         : 84,       # Each frame resized to 84×84
    'n_actions'          : 5,        # Discrete action space

    # Training
    'total_episodes'     : 500,
    'max_steps'          : 1000,     # Max environment steps per episode
    'batch_size'         : 64,
    'gamma'              : 0.99,     # Discount factor
    'lr'                 : 1e-4,     # Adam learning rate
    'min_replay_size'    : 5_000,    # Warmup steps before training begins
    'replay_buffer_size' : 50_000,

    # Epsilon-greedy exploration
    'eps_start'          : 1.0,
    'eps_end'            : 0.05,
    'eps_decay_steps'    : 100_000,

    # Target network
    'target_update_freq' : 1_000,   # Steps between target-net syncs

    # Logging & checkpointing
    'save_freq'          : 50,      # Save checkpoint every N episodes
    'plot_freq'          : 25,      # Refresh reward plot every N episodes
    'solve_score'        : 700,     # 100-episode average considered "solved"
}

# Discrete action map: 0=noop 1=left 2=right 3=gas 4=brake
ACTIONS = [
    [0, 0, 0],   # no-op
    [-1, 0, 0],  # steer left
    [1,  0, 0],  # steer right
    [0,  1, 0],  # accelerate
    [0,  0, 0.8],# brake
]

print('✓ Config loaded.')

## Step 3 — Frame preprocessing

In [ ]:
def preprocess_frame(frame: np.ndarray) -> np.ndarray:
    """
    Convert a raw 96×96 RGB frame to a normalised 84×84 grayscale float.

    Args:
        frame: uint8 array of shape (96, 96, 3)
    Returns:
        float32 array of shape (84, 84), values in [0, 1]
    """
    gray    = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
    resized = cv2.resize(gray, (CONFIG['frame_size'], CONFIG['frame_size']),
                         interpolation=cv2.INTER_AREA)
    return resized.astype(np.float32) / 255.0


class FrameStack:
    """
    Maintains a sliding window of the N most-recent processed frames.

    Stacking frames provides the network with implicit velocity and
    directional information without explicit feature engineering.
    """

    def __init__(self, n_frames: int, frame_size: int):
        self.n_frames  = n_frames
        self.frame_size = frame_size
        self.frames    = collections.deque(maxlen=n_frames)

    def reset(self, frame: np.ndarray) -> np.ndarray:
        """Initialise stack by repeating the first frame N times."""
        processed = preprocess_frame(frame)
        for _ in range(self.n_frames):
            self.frames.append(processed)
        return self._state()

    def step(self, frame: np.ndarray) -> np.ndarray:
        """Push a new frame and return the updated stack."""
        self.frames.append(preprocess_frame(frame))
        return self._state()

    def _state(self) -> np.ndarray:
        """Return stacked frames as shape (n_frames, H, W)."""
        return np.array(self.frames, dtype=np.float32)


print(f'✓ Frame preprocessing defined.')
print(f'  CNN input shape: ({CONFIG["frame_stack"]}, {CONFIG["frame_size"]}, {CONFIG["frame_size"]})')

## Step 4 — CNN Q-Network

In [ ]:
class DQNNetwork(nn.Module):
    """
    Convolutional Q-Network mapping pixel observations to action Q-values.

    Architecture (DeepMind Nature DQN, Mnih et al. 2015):
        Input  : (B, 4, 84, 84)   — 4 stacked grayscale frames
        Conv1  : 32 × 8×8, stride 4  →  (B, 32, 20, 20)
        Conv2  : 64 × 4×4, stride 2  →  (B, 64,  9,  9)
        Conv3  : 64 × 3×3, stride 1  →  (B, 64,  7,  7)  [3 136 units]
        FC1    : 512 units
        FC2    : n_actions units  — one Q-value per discrete action
    """

    def __init__(self, n_frames: int, n_actions: int):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(n_frames, 32, kernel_size=8, stride=4), nn.ReLU(),
            nn.Conv2d(32,       64, kernel_size=4, stride=2), nn.ReLU(),
            nn.Conv2d(64,       64, kernel_size=3, stride=1), nn.ReLU(),
        )

        # Compute flattened conv output size dynamically (avoids hard-coding)
        with torch.no_grad():
            dummy      = torch.zeros(1, n_frames, CONFIG['frame_size'], CONFIG['frame_size'])
            conv_out   = int(np.prod(self.conv(dummy).shape))

        self.fc = nn.Sequential(
            nn.Linear(conv_out, 512), nn.ReLU(),
            nn.Linear(512, n_actions),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass. x: (B, frames, H, W) → (B, n_actions)."""
        return self.fc(self.conv(x).flatten(start_dim=1))


# Sanity check
_net    = DQNNetwork(CONFIG['frame_stack'], CONFIG['n_actions']).to(device)
_dummy  = torch.zeros(1, CONFIG['frame_stack'], CONFIG['frame_size'], CONFIG['frame_size']).to(device)
_params = sum(p.numel() for p in _net.parameters() if p.requires_grad)
print(f'✓ DQNNetwork defined.')
print(f'  Output shape      : {_net(_dummy).shape}  (batch=1, n_actions={CONFIG["n_actions"]})')
print(f'  Trainable params  : {_params:,}')
del _net, _dummy

## Step 5 — Experience Replay Buffer

In [ ]:
class ReplayBuffer:
    """
    Fixed-capacity circular buffer storing (s, a, r, s', done) transitions.

    Motivation (Mnih et al., 2015):
      1. Breaks temporal correlation between consecutive training samples.
      2. Enables reuse of rare transitions (e.g. goal-reaching events).
      3. Improves sample efficiency and training stability.
    """

    def __init__(self, capacity: int):
        self.buffer = collections.deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        """Store a single transition."""
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size: int):
        """
        Uniformly sample a mini-batch.
        Returns GPU-ready tensors.
        """
        states, actions, rewards, next_states, dones = zip(
            *random.sample(self.buffer, batch_size)
        )
        to = lambda x, dtype: torch.tensor(np.array(x), dtype=dtype, device=device)
        return (
            to(states,      torch.float32),
            to(actions,     torch.int64),
            to(rewards,     torch.float32),
            to(next_states, torch.float32),
            to(dones,       torch.float32),
        )

    def __len__(self):
        return len(self.buffer)


print(f'✓ ReplayBuffer defined.')
print(f'  Capacity  : {CONFIG["replay_buffer_size"]:,} transitions')
print(f'  Warmup    : {CONFIG["min_replay_size"]:,} steps before training')

## Step 6 — DQN Agent

In [ ]:
class DQNAgent:
    """
    DQN Agent implementing the three core components:
      1. Epsilon-greedy action selection (exploration → exploitation)
      2. Bellman target with frozen target network (training stability)
      3. Experience replay (sample efficiency)

    Bellman update:
      Q(s,a) ← Q(s,a) + α · [r + γ · max_a' Q_target(s',a') − Q(s,a)]
    """

    def __init__(self):
        self.n_actions   = CONFIG['n_actions']
        self.gamma       = CONFIG['gamma']
        self.batch_size  = CONFIG['batch_size']
        self.total_steps = 0
        self.episodes    = 0

        # Online net — updated every training step
        self.online_net = DQNNetwork(CONFIG['frame_stack'], self.n_actions).to(device)
        # Target net — periodically synced from online net; never directly trained
        self.target_net = DQNNetwork(CONFIG['frame_stack'], self.n_actions).to(device)
        self.target_net.load_state_dict(self.online_net.state_dict())
        self.target_net.eval()

        self.optimizer = optim.Adam(self.online_net.parameters(), lr=CONFIG['lr'])
        self.replay    = ReplayBuffer(CONFIG['replay_buffer_size'])
        self.losses    = []

    # ── Exploration ───────────────────────────────────────────────────────────

    @property
    def epsilon(self) -> float:
        """Linearly decay ε from eps_start → eps_end over eps_decay_steps."""
        progress = min(self.total_steps / CONFIG['eps_decay_steps'], 1.0)
        return CONFIG['eps_start'] + progress * (CONFIG['eps_end'] - CONFIG['eps_start'])

    def select_action(self, state: np.ndarray, greedy: bool = False) -> int:
        """ε-greedy action selection (set greedy=True for evaluation)."""
        if not greedy and random.random() < self.epsilon:
            return random.randrange(self.n_actions)          # explore
        with torch.no_grad():
            s = torch.FloatTensor(state).unsqueeze(0).to(device)
            return self.online_net(s).argmax(dim=1).item()   # exploit

    # ── Memory ────────────────────────────────────────────────────────────────

    def store(self, state, action, reward, next_state, done):
        self.replay.push(state, action, reward, next_state, float(done))

    # ── Learning ──────────────────────────────────────────────────────────────

    def train_step(self):
        """Sample a mini-batch and apply one Bellman gradient."""
        if len(self.replay) < CONFIG['min_replay_size']:
            return None

        states, actions, rewards, next_states, dones = self.replay.sample(self.batch_size)

        # Current Q-values for taken actions
        q_values  = self.online_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)

        # Bellman target (no gradient through target net)
        with torch.no_grad():
            max_next_q = self.target_net(next_states).max(dim=1).values
            targets    = rewards + self.gamma * max_next_q * (1 - dones)

        loss = nn.SmoothL1Loss()(q_values, targets)   # Huber loss — robust to outliers
        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.online_net.parameters(), 10)  # gradient clipping
        self.optimizer.step()

        self.losses.append(loss.item())
        return loss.item()

    def update_target(self):
        """Hard update: copy online net weights - target net."""
        self.target_net.load_state_dict(self.online_net.state_dict())

    # ── Persistence ───────────────────────────────────────────────────────────

    def save(self, path: str):
        torch.save({
            'online_net'  : self.online_net.state_dict(),
            'target_net'  : self.target_net.state_dict(),
            'optimizer'   : self.optimizer.state_dict(),
            'total_steps' : self.total_steps,
            'episodes'    : self.episodes,
        }, path)

    def load(self, path: str):
        ckpt = torch.load(path, map_location=device)
        self.online_net.load_state_dict(ckpt['online_net'])
        self.target_net.load_state_dict(ckpt['target_net'])
        self.optimizer.load_state_dict(ckpt['optimizer'])
        self.total_steps = ckpt.get('total_steps', 0)
        self.episodes    = ckpt.get('episodes',    0)
        print(f'Loaded checkpoint — episode {self.episodes}, step {self.total_steps:,}')


print('✓ DQNAgent defined.')

## Step 7 — Environment factory & reward shaping

In [ ]:
def make_env() -> gym.Env:
    """Create a discrete-action CarRacing environment."""
    return gym.make(CONFIG['env_name'], continuous=False, render_mode='rgb_array')


def shape_reward(reward: float) -> float:
    """
    Light reward shaping to discourage grass driving.

    CarRacing default: +1000/N per tile visited (N = total track tiles).
    Negative reward signals the car is on grass — That amplify this penalty
    to encourage staying on track.
    """
    return reward * 2.0 if reward < 0 else reward


# Quick environment sanity check
_env = make_env()
obs, _ = _env.reset()
print(f'✓ Environment created.')
print(f'  Observation space : {_env.observation_space}')
print(f'  Action space      : {_env.action_space}')
print(f'  Raw obs shape     : {obs.shape}')
_env.close()

## Step 8 — Plotting utilities

In [ ]:
def plot_rewards(rewards: list, avg_rewards: list, episode: int):
    """
    Render and save the training reward curve.
    Called every CONFIG['plot_freq'] episodes during training.
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle(f'DQN CarRacing-v2 — Training Progress (Episode {episode})',
                 fontsize=13, fontweight='bold')

    # Left: reward curve
    ax = axes[0]
    ax.plot(rewards,     alpha=0.35, color='#185FA5', linewidth=0.8, label='Episode reward')
    ax.plot(avg_rewards, color='#E24B4A', linewidth=2.0,             label='100-ep rolling avg')
    ax.axhline(CONFIG['solve_score'], color='#1D9E75', linestyle='--', alpha=0.8,
               label=f'Solve threshold ({CONFIG["solve_score"]})')
    ax.set_xlabel('Episode'); ax.set_ylabel('Total reward')
    ax.set_title('Reward per episode')
    ax.legend(fontsize=9); ax.grid(alpha=0.25)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

    # Right: epsilon decay
    ax2 = axes[1]
    steps = min(episode * CONFIG['max_steps'], CONFIG['eps_decay_steps'] + 1)
    eps_curve = [
        max(CONFIG['eps_end'],
            CONFIG['eps_start'] + min(i / CONFIG['eps_decay_steps'], 1.0)
            * (CONFIG['eps_end'] - CONFIG['eps_start']))
        for i in range(steps)
    ]
    ax2.plot(eps_curve, color='#EF9F27', linewidth=1.5)
    ax2.fill_between(range(steps), eps_curve, alpha=0.15, color='#EF9F27')
    ax2.set_xlabel('Training step'); ax2.set_ylabel('ε')
    ax2.set_title('Exploration rate (ε) decay')
    ax2.grid(alpha=0.25)
    ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.savefig('plots/reward_curve.png', dpi=130, bbox_inches='tight')
    plt.close()


print('✓ Plotting utilities defined.')

## Step 9 — Training loop

> **Expected time on Colab T4:** ~3–5 hours for 1 000 episodes.  
> Episodes 1–100: random behaviour · 100–300: starts following track · 300+: consistent laps.  
> Checkpoints saved every 50 episodes — safe to resume if Colab disconnects.

In [ ]:
def train(resume_from: str = None):
    """
    Main DQN training loop.
    """
    env         = make_env()
    agent       = DQNAgent()
    frame_stack = FrameStack(CONFIG['frame_stack'], CONFIG['frame_size'])

    if resume_from:
        agent.load(resume_from)

    episode_rewards  = []
    avg_rewards      = []
    best_avg_reward  = -float('inf')
    start_time       = time.time()

    print('=' * 68)
    print(f'  DQN Training  |  {CONFIG["env_name"]}  |  Device: {device}')
    print(f'  Episodes: {CONFIG["total_episodes"]}  |  Warmup: {CONFIG["min_replay_size"]:,} steps')
    print('=' * 68)

    for episode in range(1, CONFIG['total_episodes'] + 1):
        obs, _  = env.reset()
        state   = frame_stack.reset(obs)
        ep_reward, ep_loss, loss_count = 0.0, 0.0, 0

        for _ in range(CONFIG['max_steps']):
            action_idx              = agent.select_action(state)
            obs, reward, term, trunc, _ = env.step(ACTIONS[action_idx])
            done                    = term or trunc
            next_state              = frame_stack.step(obs)

            agent.store(state, action_idx, shape_reward(reward), next_state, done)

            loss = agent.train_step()
            if loss is not None:
                ep_loss   += loss
                loss_count += 1

            agent.total_steps += 1
            if agent.total_steps % CONFIG['target_update_freq'] == 0:
                agent.update_target()

            ep_reward += reward
            state      = next_state
            if done:
                break

        agent.episodes += 1
        episode_rewards.append(ep_reward)
        rolling_avg = np.mean(episode_rewards[-100:])
        avg_rewards.append(rolling_avg)

        avg_loss = ep_loss / loss_count if loss_count else 0.0
        elapsed  = (time.time() - start_time) / 60

        print(
            f'Ep {episode:4d} | '
            f'Reward {ep_reward:8.1f} | '
            f'Avg(100) {rolling_avg:7.1f} | '
            f'ε {agent.epsilon:.3f} | '
            f'Loss {avg_loss:.4f} | '
            f'Steps {agent.total_steps:,} | '
            f'{elapsed:.1f}m'
        )

        # Save best model
        if episode >= 100 and rolling_avg > best_avg_reward:
            best_avg_reward = rolling_avg
            agent.save('checkpoints/best_model.pth')
            print(f'  ★ New best avg: {rolling_avg:.1f} → best_model.pth')

        # Periodic checkpoint
        if episode % CONFIG['save_freq'] == 0:
            agent.save(f'checkpoints/episode_{episode:04d}.pth')

        # Refresh plot
        if episode % CONFIG['plot_freq'] == 0:
            plot_rewards(episode_rewards, avg_rewards, episode)

        # Early stop if solved
        if rolling_avg >= CONFIG['solve_score']:
            print(f'\n🏆 Solved at episode {episode}! Avg reward: {rolling_avg:.1f}')
            agent.save('checkpoints/solved_model.pth')
            break

    env.close()
    plot_rewards(episode_rewards, avg_rewards, agent.episodes)
    print(f'\n✓ Training complete. Best avg reward: {best_avg_reward:.1f}')
    return agent, episode_rewards


# ── Run training ──────────────────────────────────────────────────────────────
# To resume: train(resume_from='checkpoints/episode_0200.pth')
agent, rewards = train()

## Step 10 — Evaluate trained agent

In [ ]:
def evaluate(model_path: str = 'checkpoints/best_model.pth',
             n_episodes: int = 10) -> list:
    """
    Benchmark the trained agent with greedy policy (ε=0).

    Args:
        model_path : Path to saved checkpoint.
        n_episodes : Number of evaluation episodes.
    Returns:
        List of per-episode rewards.
    """
    agent       = DQNAgent()
    agent.load(model_path)
    agent.online_net.eval()

    env         = make_env()
    frame_stack = FrameStack(CONFIG['frame_stack'], CONFIG['frame_size'])
    eval_rewards = []

    print(f'Evaluating {n_episodes} episodes (greedy policy, no exploration)...')
    for ep in range(1, n_episodes + 1):
        obs, _  = env.reset()
        state   = frame_stack.reset(obs)
        total_r = 0.0
        done    = False

        while not done:
            action_idx              = agent.select_action(state, greedy=True)
            obs, r, term, trunc, _  = env.step(ACTIONS[action_idx])
            done                    = term or trunc
            state                   = frame_stack.step(obs)
            total_r                += r

        eval_rewards.append(total_r)
        print(f'  Episode {ep:2d}: {total_r:8.1f}')

    env.close()
    print(f'\nMean reward : {np.mean(eval_rewards):.1f}')
    print(f'Std reward  : {np.std(eval_rewards):.1f}')
    return eval_rewards


eval_rewards = evaluate('checkpoints/best_model.pth', n_episodes=10)

## Step 11 — Record a video of the trained agent

In [ ]:
def record_video(model_path: str = 'checkpoints/best_model.pth'):

    agent       = DQNAgent()
    agent.load(model_path)
    agent.online_net.eval()

    env = gym.make(CONFIG['env_name'], continuous=False, render_mode='rgb_array')
    env = RecordVideo(env, video_folder='videos',
                      name_prefix='dqn_carracing',
                      episode_trigger=lambda ep: True)

    frame_stack = FrameStack(CONFIG['frame_stack'], CONFIG['frame_size'])
    obs, _      = env.reset()
    state       = frame_stack.reset(obs)
    total_r     = 0.0
    done        = False

    while not done:
        action_idx              = agent.select_action(state, greedy=True)
        obs, r, term, trunc, _  = env.step(ACTIONS[action_idx])
        done                    = term or trunc
        state                   = frame_stack.step(obs)
        total_r                += r

    env.close()
    print(f'✓ Video saved to videos/  |  Episode reward: {total_r:.1f}')
    print('  → Download via Colab file browser (left sidebar) or the cell below.')


record_video('checkpoints/best_model.pth')

## Step 12 — Download all outputs

In [ ]:
# Zip everything and trigger browser download
!zip -r dqn_outputs.zip checkpoints/ plots/ videos/ -q

from google.colab import files
files.download('dqn_outputs.zip')

print('✓ Downloaded dqn_outputs.zip')
print()
print('GitHub repo structure:')
print('  DQN_CarRacing.ipynb          ← this notebook (root)')
print('  checkpoints/best_model.pth   ← checkpoints/')
print('  plots/reward_curve.png       ← docs/reward_curve.png')
print('  videos/*.mp4                 ← docs/demo.mp4')

---
## Summary

| Component | Class / Function | Description |
|-----------|-----------------|-------------|
| Frame preprocessing | `preprocess_frame`, `FrameStack` | Grayscale + resize + temporal stacking |
| Q-Network | `DQNNetwork` | 3-layer CNN + 2-layer FC |
| Replay buffer | `ReplayBuffer` | 50k-capacity circular buffer |
| Agent | `DQNAgent` | ε-greedy + Bellman update + target net |
| Training | `train()` | 1 000 episodes, checkpointing, early stop |
| Evaluation | `evaluate()` | Greedy policy benchmark |
| Video | `record_video()` | MP4 export via `RecordVideo` |

**Reference:** Mnih, V. et al. (2015). Human-level control through deep reinforcement learning. *Nature*, 518, 529–533.  
**Author:** Herin Bhatt · MSc AI/ML Independent project · 2025/2026